In [1]:
# ==============================================================================
# [GLOBAL CELL 1] KONFIGURASI SENTRAL, SEEDING, & SETUP (TRAINING NOTEBOOK)
# ==============================================================================
import os
import json
import sqlite3
import random
from enum import Enum
from typing import Tuple, Dict, Any

import numpy as np
import tensorflow as tf

# ------------------------------------------------------------------------------
# 1. ENUMERASI ARSITEKTUR (Standar 3 & 9)
# ------------------------------------------------------------------------------
class ModelArch(str, Enum):
    MOBILENET_V3 = 'MobileNetV3Small'
    EFFICIENTNET_B0 = 'EfficientNetB0'
    RESNET_50 = 'ResNet50'

# ------------------------------------------------------------------------------
# 2. SENTRALISASI KONFIGURASI (Standar 2 & 4)
# ------------------------------------------------------------------------------
class TrainingConfig:
    """Konfigurasi utama untuk Fase 1 (Head) dan Fase 2 (Fine-Tuning SGDR)."""
    
    # Dataset & Dimensi
    PATH_TRAIN_FFB: str = "/kaggle/input/datasets/marrioqqqq/dataset-sawit/dataset-sawit/train"
    PATH_VAL_FFB: str = "/kaggle/input/datasets/marrioqqqq/dataset-sawit/dataset-sawit/val"
    IMG_SIZE: Tuple[int, int] = (224, 224)
    NUM_CLASSES: int = 3
    
    # # Hyperparameter Training Statis
    # EPOCHS_PHASE1: int = 20
    # EPOCHS_PHASE2: int = 100
    # PATIENCE_EARLY_STOPPING: int = 12
    SGDR_MULTIPLIER: float = 0.1  # Base LR Fase 2 adalah 10% dari Best LR HPO

    # Hyperparameter Training Statis (MODE DEBUG)
    EPOCHS_PHASE1: int = 1
    EPOCHS_PHASE2: int = 2  # Dibuat 2 agar SGDR sempat mencatat pergerakan
    PATIENCE_EARLY_STOPPING: int = 12
    
    # Direktori HPO (Sumber Best Params)
    PATH_HPO_OUTPUTS: str = "/kaggle/input/datasets/marrioqqqq/hpo-output" # Asumsi Anda mengimpor dataset HPO ke sini
    
    # Direktori Output Training Dinamis (Standar 4)
    BASE_OUTPUT_DIR: str = "/kaggle/working/training_outputs"

    @classmethod
    def get_output_dir(cls, arch: ModelArch) -> str:
        target_dir = os.path.join(cls.BASE_OUTPUT_DIR, arch.value)
        os.makedirs(target_dir, exist_ok=True)
        return target_dir

    @classmethod
    def get_hpo_params_path(cls, arch: ModelArch) -> str:
        """Mengambil JSON parameter terbaik hasil dari Notebook HPO sebelumnya."""
        # Sesuaikan struktur folder HPO Anda jika berbeda
        return os.path.join(cls.PATH_HPO_OUTPUTS, arch.value, f"best_params_{arch.value}.json")

    @classmethod
    def get_db_path(cls, arch: ModelArch) -> str:
        return os.path.join(cls.get_output_dir(arch), f"training_log_{arch.value}.db")

    @classmethod
    def get_checkpoint_path(cls, arch: ModelArch, phase: int) -> str:
        """Standar 6: Checkpointing (.keras format) dinamis berdasarkan Fase."""
        return os.path.join(cls.get_output_dir(arch), f"model_{arch.value}_phase{phase}.keras")

# ------------------------------------------------------------------------------
# 3. REPRODUKSIBILITAS (Standar 3)
# ------------------------------------------------------------------------------
def set_deterministic_seed(seed: int = 42) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()
    print(f"[SETUP] Deterministic Seed dikunci pada: {seed}")

# ------------------------------------------------------------------------------
# 4. INISIALISASI LINGKUNGAN
# ------------------------------------------------------------------------------
set_deterministic_seed(42)

try:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print("[SETUP] Mixed Precision (FP16) Aktif untuk akselerasi.")
except Exception as e:
    print(f"[WARNING] Mixed Precision dinonaktifkan: {e}")

print(f"[SETUP] TensorFlow Version: {tf.__version__}")

[SETUP] Deterministic Seed dikunci pada: 42
[SETUP] Mixed Precision (FP16) Aktif untuk akselerasi.
[SETUP] TensorFlow Version: 2.20.0


In [2]:
# ==============================================================================
# [CELL 2] FUNCTIONAL GRAPH BUILDER & SQLITE LOGGER CALLBACK
# ==============================================================================
from tensorflow.keras import layers, models, applications, callbacks

# ------------------------------------------------------------------------------
# 1. SQLITE CALLBACK UNTUK LOGGING (Standar 8)
# ------------------------------------------------------------------------------
class SQLiteTrainingLogger(callbacks.Callback):
    """
    Custom Keras Callback menggantikan CSVLogger. 
    Merekam metrik per epoch langsung ke database SQLite untuk kueri tingkat lanjut.
    """
    def __init__(self, db_path: str, phase: int):
        super().__init__()
        self.db_path = db_path
        self.phase = phase
        self._initialize_db()

    def _initialize_db(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS training_logs (
                phase INTEGER, epoch INTEGER,
                loss REAL, accuracy REAL,
                val_loss REAL, val_accuracy REAL,
                learning_rate REAL
            )
        ''')
        conn.commit()
        conn.close()

    def on_epoch_end(self, epoch: int, logs: Dict[str, Any] = None):
        logs = logs or {}
        # Ekstrak learning rate jika scheduler digunakan
        lr = self.model.optimizer.learning_rate
        if isinstance(lr, tf.keras.optimizers.schedules.LearningRateSchedule):
            lr_value = float(lr(self.model.optimizer.iterations).numpy())
        else:
            lr_value = float(lr.numpy() if hasattr(lr, 'numpy') else lr)

        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute('''
            INSERT INTO training_logs 
            (phase, epoch, loss, accuracy, val_loss, val_accuracy, learning_rate)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        ''', (
            self.phase, epoch + 1,
            logs.get('loss', 0.0), logs.get('accuracy', 0.0),
            logs.get('val_loss', 0.0), logs.get('val_accuracy', 0.0),
            lr_value
        ))
        conn.commit()
        conn.close()

# ------------------------------------------------------------------------------
# 2. BASE CLASS: FUNCTIONAL BUILDER AGNOSTIK (Standar 5 & 7)
# ------------------------------------------------------------------------------
class FFBAgnosticBuilder:
    """
    Membangun graf fungsional yang memastikan backbone memiliki nama statis ('agnostic_backbone')
    sehingga proses Fine-Tuning/Unfreezing di Fase 2 tidak perlu mempedulikan arsitektur apa yang dipakai.
    """
    def __init__(self, config: TrainingConfig, architecture: ModelArch):
        self.cfg = config
        self.arch = architecture

    def build_phase1_graph(self, dropout_rate: float) -> models.Model:
        """Merakit graf dengan backbone beku untuk Transfer Learning (Fase 1)."""
        inputs = tf.keras.Input(shape=(self.cfg.IMG_SIZE[0], self.cfg.IMG_SIZE[1], 3), name="input_ffb")
        
        # Augmentasi
        x = tf.keras.Sequential([
            layers.RandomFlip("horizontal_and_vertical"),
            layers.RandomRotation(0.3),
            layers.RandomTranslation(0.1, 0.1),
            layers.RandomZoom(0.2),
            layers.RandomBrightness(0.2),
            layers.RandomContrast(0.2),
        ], name="augmentation_block")(inputs)

        # Injeksi Backbone dengan nama eksplisit AGNOSTIK
        if self.arch == ModelArch.MOBILENET_V3:
            backbone = applications.MobileNetV3Small(input_tensor=x, include_top=False, weights='imagenet')
        elif self.arch == ModelArch.EFFICIENTNET_B0:
            backbone = applications.EfficientNetB0(input_tensor=x, include_top=False, weights='imagenet')
        elif self.arch == ModelArch.RESNET_50:
            x = layers.Lambda(applications.resnet50.preprocess_input, name="resnet_preprocess")(x)
            backbone = applications.ResNet50(input_tensor=x, include_top=False, weights='imagenet')
        else:
            raise ValueError("Arsitektur tidak didukung.")

        # Menghapus sifat trainable backbone (Fase 1) dan menamai ulang nama layer utamanya
        backbone.trainable = False
        backbone._name = "agnostic_backbone" # KUNCI AGNOSTIK (Standar 5)
        
        x = backbone.output

        # Classifier Head
        x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
        x = layers.BatchNormalization(name="head_bn")(x)
        x = layers.Dropout(dropout_rate, name="head_dropout")(x)
        outputs = layers.Dense(self.cfg.NUM_CLASSES, activation='softmax', 
                               name=f"{self.arch.value}_output", dtype='float32')(x)

        return models.Model(inputs=inputs, outputs=outputs, name=f"Edge_{self.arch.value}")

print("[BERHASIL] Kelas Config, SQLite Logger, dan Agnostic Builder siap digunakan.")

[BERHASIL] Kelas Config, SQLite Logger, dan Agnostic Builder siap digunakan.


In [3]:
# ==============================================================================
# [CELL 3] OOP TRAINING ENGINE (FASE 1 & FASE 2)
# ==============================================================================
import os
import json
import tensorflow as tf
from tensorflow.keras import optimizers, callbacks, models

class FFBTrainingEngine(FFBAgnosticBuilder):
    """
    Standard 9: Base Class yang mewarisi fungsi pembuatan graf dari FFBAgnosticBuilder.
    Bertugas memuat dataset, mengaplikasikan Best Params dari HPO, dan mengeksekusi
    pelatihan Fase 1 maupun Fase 2 secara dinamis.
    """
    def __init__(self, config: TrainingConfig, architecture: ModelArch):
        super().__init__(config, architecture)
        
        # Standar 9: Validasi JSON Pra-Eksekusi
        self.hpo_params_path = self.cfg.get_hpo_params_path(self.arch)
        if not os.path.exists(self.hpo_params_path):
            raise FileNotFoundError(f"[ERROR] JSON Best Params tidak ditemukan di: {self.hpo_params_path}")
            
        with open(self.hpo_params_path, 'r') as f:
            self.best_params = json.load(f)
            
        print(f"\n[INFO] Engine {self.arch.value} Siap! Best Params Termuat: {self.best_params}")
        
        # Persiapan Dataset Dinamis (menggunakan batch_size dari HPO)
        self._build_datasets()

    def _build_datasets(self) -> None:
        """Membangun pipeline dataset dengan caching untuk efisiensi I/O disk."""
        batch_size = self.best_params['batch_size']
        
        self.ds_train = tf.keras.utils.image_dataset_from_directory(
            self.cfg.PATH_TRAIN_FFB, image_size=self.cfg.IMG_SIZE, 
            batch_size=batch_size, shuffle=True, label_mode='int', verbose=0
        ).cache().take(1).prefetch(tf.data.AUTOTUNE)
        
        self.ds_val = tf.keras.utils.image_dataset_from_directory(
            self.cfg.PATH_VAL_FFB, image_size=self.cfg.IMG_SIZE, 
            batch_size=batch_size, shuffle=False, label_mode='int', verbose=0
        ).cache().take(1).prefetch(tf.data.AUTOTUNE)
        
        self.steps_per_epoch = len(self.ds_train)

    # --------------------------------------------------------------------------
    # FASE 1: TRANSFER LEARNING (HEAD ONLY)
    # --------------------------------------------------------------------------
    def run_phase1(self) -> None:
        print("\n" + "="*60)
        print(f"🚀 MEMULAI FASE 1: TRANSFER LEARNING ({self.arch.value})")
        print("="*60)
        
        # 1. Bangun Graf (Dari Base Class)
        model = self.build_phase1_graph(dropout_rate=self.best_params['dropout_rate'])
        
        # 2. Setup Optimizer sesuai temuan HPO
        opt_str = self.best_params['optimizer']
        lr = self.best_params['learning_rate']
        wd = self.best_params['weight_decay']
        
        if opt_str == 'adamw':
            opt = optimizers.AdamW(learning_rate=lr, weight_decay=wd)
        elif opt_str == 'adam':
            opt = optimizers.Adam(learning_rate=lr, weight_decay=wd)
        else:
            opt = optimizers.SGD(learning_rate=lr, momentum=0.9, weight_decay=wd)
            
        model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        
        # 3. Eksekusi Pelatihan
        checkpoint_path = self.cfg.get_checkpoint_path(self.arch, phase=1)
        db_path = self.cfg.get_db_path(self.arch)
        
        callbacks_list = [
            callbacks.ModelCheckpoint(filepath=checkpoint_path, save_best_only=True, monitor='val_loss', mode='min', verbose=1),
            SQLiteTrainingLogger(db_path=db_path, phase=1)
        ]
        
        model.fit(self.ds_train, validation_data=self.ds_val, epochs=self.cfg.EPOCHS_PHASE1, callbacks=callbacks_list)
        print(f"\n[BERHASIL] Model Fase 1 Tersimpan di: {checkpoint_path}")

    # --------------------------------------------------------------------------
    # FASE 2: FINE-TUNING + SGDR (UNFREEZE BACKBONE)
    # --------------------------------------------------------------------------
    def run_phase2(self) -> None:
        print("\n" + "="*60)
        print(f"🔥 MEMULAI FASE 2: FINE-TUNING SGDR ({self.arch.value})")
        print("="*60)
        
        # 1. Muat Model Terbaik Fase 1
        checkpoint_p1 = self.cfg.get_checkpoint_path(self.arch, phase=1)
        if not os.path.exists(checkpoint_p1):
            raise FileNotFoundError(f"[ERROR] Model Fase 1 tidak ditemukan: {checkpoint_p1}")
            
        model = models.load_model(checkpoint_p1)
        
        # 2. STANDAR 5 (Agnostik): Buka Gembok Berdasarkan Topologi Layer
        # Strategi baru: Mencari layer yang berbentuk nested Keras Model
        backbone = None
        for layer in model.layers:
            if isinstance(layer, tf.keras.Model):
                backbone = layer
                break
                
        if backbone is None:
            raise ValueError("[ERROR] Backbone (nested Keras Model) tidak ditemukan di dalam graf!")
            
        backbone.trainable = True
        
        # Proteksi Layer BatchNormalization
        frozen_bn = 0
        for layer in backbone.layers:
            if isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False
                frozen_bn += 1
                
        print(f"[INFO] Backbone Dibuka ({len(backbone.layers)} layer). Layer BN Dilindungi: {frozen_bn}.")
        
        # 3. Setup SGDR (Cosine Annealing Warm Restarts)
        base_lr_p2 = self.best_params['learning_rate'] * self.cfg.SGDR_MULTIPLIER
        lr_scheduler = optimizers.schedules.CosineDecayRestarts(
            initial_learning_rate=base_lr_p2,
            first_decay_steps=self.steps_per_epoch * 10,
            t_mul=2.0, m_mul=1.0, alpha=1e-3
        )
        
        # Menggunakan SGD untuk SGDR
        opt = optimizers.SGD(learning_rate=lr_scheduler, momentum=0.9, weight_decay=self.best_params['weight_decay'])
        model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        
        # 4. Eksekusi Pelatihan
        checkpoint_p2 = self.cfg.get_checkpoint_path(self.arch, phase=2)
        db_path = self.cfg.get_db_path(self.arch)
        
        callbacks_list = [
            callbacks.ModelCheckpoint(filepath=checkpoint_p2, save_best_only=True, monitor='val_loss', mode='min', verbose=1),
            callbacks.EarlyStopping(monitor='val_loss', patience=self.cfg.PATIENCE_EARLY_STOPPING, restore_best_weights=True, verbose=1),
            SQLiteTrainingLogger(db_path=db_path, phase=2)
        ]
        
        # Initial epoch memastikan nomor grafik di TensorBoard/SQLite menyambung dengan Fase 1
        model.fit(
            self.ds_train, validation_data=self.ds_val, 
            initial_epoch=self.cfg.EPOCHS_PHASE1, 
            epochs=self.cfg.EPOCHS_PHASE1 + self.cfg.EPOCHS_PHASE2, 
            callbacks=callbacks_list
        )
        print(f"\n[BERHASIL] Model Final FP32 Tersimpan di: {checkpoint_p2}")
        
print("[BERHASIL] Cell 3 (Training Engine) dimuat. Siap dieksekusi!")

[BERHASIL] Cell 3 (Training Engine) dimuat. Siap dieksekusi!


In [4]:
# ==============================================================================
# [CELL 4] EKSEKUTOR FASE 1: TRANSFER LEARNING (LOOP RUNNER)
# ==============================================================================
import gc
import tensorflow as tf

config = TrainingConfig()

# ------------------------------------------------------------------------------
# MODE SELEKSI ARSITEKTUR
# Silakan beri komentar (#) pada model yang tidak ingin dilatih saat ini.
# ------------------------------------------------------------------------------
MODELS_TO_TRAIN = [
    ModelArch.MOBILENET_V3, 
    # ModelArch.EFFICIENTNET_B0, 
    # ModelArch.RESNET_50
]

print("\n" + "★"*75)
print(f"🔥 MEMULAI PELATIHAN FASE 1 UNTUK {len(MODELS_TO_TRAIN)} ARSITEKTUR")
print("★"*75)

for idx, arch in enumerate(MODELS_TO_TRAIN, start=1):
    print(f"\n{'='*75}")
    print(f"[{idx}/{len(MODELS_TO_TRAIN)}] EKSEKUSI FASE 1: {arch.value}")
    print(f"{'='*75}")
    
    try:
        # 1. Reset State & Kunci Seed Spesifik untuk Iterasi Ini (Standar 3)
        tf.keras.backend.clear_session()
        gc.collect()
        set_deterministic_seed(42) 
        
        # 2. Inisialisasi Engine & Eksekusi
        engine = FFBTrainingEngine(config=config, architecture=arch)
        engine.run_phase1()
        
        # 3. Cleanup Ekstrem untuk Mencegah Kebocoran Memori (OOM)
        print(f"\n[CLEANUP] Membersihkan VRAM GPU pasca-Fase 1 untuk {arch.value}...")
        del engine
        gc.collect()
        
    except Exception as e:
        print(f"\n[CRITICAL ERROR] Eksekusi Fase 1 gagal pada {arch.value}: {e}")
        # Jika satu model gagal, loop akan melompat ke model berikutnya, bukan mati total.

print("\n" + "★"*75)
print("🏆 SELURUH PELATIHAN FASE 1 SELESAI!")
print("★"*75)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🔥 MEMULAI PELATIHAN FASE 1 UNTUK 1 ARSITEKTUR
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

[1/1] EKSEKUSI FASE 1: MobileNetV3Small
[SETUP] Deterministic Seed dikunci pada: 42

[CRITICAL ERROR] Eksekusi Fase 1 gagal pada MobileNetV3Small: [ERROR] JSON Best Params tidak ditemukan di: /kaggle/input/datasets/marrioqqqq/hpo-output/MobileNetV3Small/best_params_MobileNetV3Small.json

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🏆 SELURUH PELATIHAN FASE 1 SELESAI!
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★


In [5]:
# ==============================================================================
# [CELL 5] EKSEKUTOR FASE 2: FINE-TUNING DENGAN SGDR (LOOP RUNNER)
# ==============================================================================
import gc
import tensorflow as tf

config = TrainingConfig()

# ------------------------------------------------------------------------------
# MODE SELEKSI ARSITEKTUR (Pastikan model ini sudah lulus Fase 1)
# ------------------------------------------------------------------------------
MODELS_TO_TRAIN_PHASE2 = [
    ModelArch.MOBILENET_V3, 
    # ModelArch.EFFICIENTNET_B0, 
    # ModelArch.RESNET_50
]

print("\n" + "★"*75)
print(f"🔥 MEMULAI PELATIHAN FASE 2 (SGDR) UNTUK {len(MODELS_TO_TRAIN_PHASE2)} ARSITEKTUR")
print("★"*75)

for idx, arch in enumerate(MODELS_TO_TRAIN_PHASE2, start=1):
    print(f"\n{'='*75}")
    print(f"[{idx}/{len(MODELS_TO_TRAIN_PHASE2)}] EKSEKUSI FASE 2: {arch.value}")
    print(f"{'='*75}")
    
    try:
        # 1. Reset State & Kunci Seed
        tf.keras.backend.clear_session()
        gc.collect()
        set_deterministic_seed(42)
        
        # 2. Inisialisasi Engine (Engine otomatis mendeteksi model Fase 1)
        engine = FFBTrainingEngine(config=config, architecture=arch)
        engine.run_phase2()
        
        # 3. Cleanup
        print(f"\n[CLEANUP] Membersihkan VRAM GPU pasca-Fase 2 untuk {arch.value}...")
        del engine
        gc.collect()
        
    except Exception as e:
        print(f"\n[CRITICAL ERROR] Eksekusi Fase 2 gagal pada {arch.value}: {e}")

print("\n" + "★"*75)
print("🏆 SELURUH PELATIHAN FASE 2 (FINAL FP32) SELESAI!")
print("★"*75)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🔥 MEMULAI PELATIHAN FASE 2 (SGDR) UNTUK 1 ARSITEKTUR
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

[1/1] EKSEKUSI FASE 2: MobileNetV3Small
[SETUP] Deterministic Seed dikunci pada: 42

[CRITICAL ERROR] Eksekusi Fase 2 gagal pada MobileNetV3Small: [ERROR] JSON Best Params tidak ditemukan di: /kaggle/input/datasets/marrioqqqq/hpo-output/MobileNetV3Small/best_params_MobileNetV3Small.json

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🏆 SELURUH PELATIHAN FASE 2 (FINAL FP32) SELESAI!
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★


In [6]:
# ==============================================================================
# [CELL 6] OOP TRAINING VISUALIZER: CLASS DEFINITION (SQLITE BACKEND)
# ==============================================================================
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, FileLink

class TrainingVisualizer:
    """
    Standard 9: Base Class Agnostik untuk merender grafik dari SQLite Training Log.
    Menghasilkan visualisasi Akurasi, Loss, dan fluktuasi Learning Rate (SGDR).
    """
    def __init__(self, config: TrainingConfig, architecture: ModelArch):
        self.cfg = config
        self.arch = architecture
        self.db_path = self.cfg.get_db_path(self.arch)
        
        # Standar 4: Isolasi Folder Visualisasi Khusus
        self.vis_dir = os.path.join(self.cfg.get_output_dir(self.arch), "visualisation")
        os.makedirs(self.vis_dir, exist_ok=True)
        
        # Path File Ekspor
        self.p_metrics = os.path.join(self.vis_dir, f"train_01_{self.arch.value}_metrics_history.png")
        self.p_lr = os.path.join(self.vis_dir, f"train_02_{self.arch.value}_learning_rate_sgdr.png")
        
        print(f"\n" + "="*60)
        print(f"📊 INISIALISASI VISUALIZER TRAINING: {self.arch.value}")
        print("="*60)
        
        self._validate_database()
        self.df = self._load_data()

    # --------------------------------------------------------------------------
    # UNIT TESTING & DATA EXTRACTION (Standar 8 & 9)
    # --------------------------------------------------------------------------
    def _validate_database(self) -> None:
        """Memastikan database memiliki data riwayat training sebelum plotting."""
        if not os.path.exists(self.db_path):
            raise FileNotFoundError(f"[ERROR] Database {self.db_path} tidak ditemukan.")
        if os.path.getsize(self.db_path) == 0:
            raise ValueError(f"[ERROR] Database {self.db_path} kosong.")
            
    def _load_data(self) -> pd.DataFrame:
        """Membaca log dari SQLite dan mengonversinya ke dalam DataFrame."""
        conn = sqlite3.connect(self.db_path)
        df = pd.read_sql_query("SELECT * FROM training_logs ORDER BY phase, epoch", conn)
        conn.close()
        
        if df.empty:
            raise ValueError(f"[ERROR] Tabel 'training_logs' kosong pada arsitektur {self.arch.value}.")
        return df

    # --------------------------------------------------------------------------
    # FUNGSI RENDER: METRIK ACCURACY & LOSS
    # --------------------------------------------------------------------------
    def plot_training_metrics(self) -> None:
        """Merender grafik Loss & Accuracy, dengan garis pemisah antara Fase 1 dan Fase 2."""
        plt.style.use('default')
        sns.set_theme(style="ticks", context="notebook")
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Identifikasi batas Epoch Fase 1 (Untuk garis putus-putus transisi)
        phase1_epochs = self.df[self.df['phase'] == 1]['epoch'].max()
        if pd.isna(phase1_epochs):
            phase1_epochs = self.cfg.EPOCHS_PHASE1 # Fallback jika log terhapus

        # Subplot 1: Akurasi
        ax1.plot(self.df['epoch'], self.df['accuracy'], label='Train Accuracy', color='#2ecc71', linewidth=2)
        ax1.plot(self.df['epoch'], self.df['val_accuracy'], label='Val Accuracy', color='#27ae60', linewidth=2.5, linestyle='--')
        ax1.axvline(x=phase1_epochs, color='red', linestyle=':', linewidth=2, label='Phase 2 Start')
        ax1.set_title(f"Accuracy History ({self.arch.value})", fontsize=12, fontweight='bold')
        ax1.set_xlabel('Epoch', fontweight='bold')
        ax1.set_ylabel('Accuracy', fontweight='bold')
        ax1.legend()
        ax1.grid(True, linestyle='--', alpha=0.6)

        # Subplot 2: Loss
        ax2.plot(self.df['epoch'], self.df['loss'], label='Train Loss', color='#e74c3c', linewidth=2)
        ax2.plot(self.df['epoch'], self.df['val_loss'], label='Val Loss', color='#c0392b', linewidth=2.5, linestyle='--')
        ax2.axvline(x=phase1_epochs, color='red', linestyle=':', linewidth=2, label='Phase 2 Start')
        ax2.set_title(f"Loss History ({self.arch.value})", fontsize=12, fontweight='bold')
        ax2.set_xlabel('Epoch', fontweight='bold')
        ax2.set_ylabel('Loss', fontweight='bold')
        ax2.legend()
        ax2.grid(True, linestyle='--', alpha=0.6)

        plt.tight_layout()
        plt.savefig(self.p_metrics, dpi=300)
        plt.close()

    # --------------------------------------------------------------------------
    # FUNGSI RENDER: LEARNING RATE SGDR (COSINE ANNEALING)
    # --------------------------------------------------------------------------
    def plot_learning_rate(self) -> None:
        """Merender grafik fluktuasi Learning Rate untuk memantau efek SGDR di Fase 2."""
        plt.figure(figsize=(10, 4))
        
        # Garis plot LR
        plt.plot(self.df['epoch'], self.df['learning_rate'], color='#2980b9', linewidth=2)
        
        phase1_epochs = self.df[self.df['phase'] == 1]['epoch'].max()
        if not pd.isna(phase1_epochs):
            plt.axvline(x=phase1_epochs, color='red', linestyle=':', linewidth=2, label='Fine-Tuning SGDR Starts')
            
        plt.title(f"Learning Rate Schedule ({self.arch.value})", fontsize=14, fontweight='bold', pad=15)
        plt.xlabel('Epoch', fontsize=12, fontweight='bold')
        plt.ylabel('Learning Rate', fontsize=12, fontweight='bold')
        plt.yscale('log')
        plt.grid(True, which="both", linestyle='--', alpha=0.6)
        plt.legend()
        
        plt.tight_layout()
        plt.savefig(self.p_lr, dpi=300)
        plt.close()

    def run_all_visualisations(self) -> None:
        self.plot_training_metrics()
        self.plot_learning_rate()
        print(f"[BERHASIL] Gambar Resolusi Tinggi Tersimpan di: {self.vis_dir}")
        display(FileLink(self.p_metrics))
        display(FileLink(self.p_lr))

print("[BERHASIL] Cell 6 (Class TrainingVisualizer) dimuat.")

[BERHASIL] Cell 6 (Class TrainingVisualizer) dimuat.


In [7]:
# ==============================================================================
# [CELL 7] EKSEKUTOR VISUALISASI TRAINING (BATCH RUNNER)
# ==============================================================================

# Menggunakan MODELS_TO_TRAIN_PHASE2 dari Cell 5 sebelumnya jika ada,
# jika dieksekusi terpisah, gunakan fallback arsitektur yang diinginkan.
try:
    models_to_visualize = MODELS_TO_TRAIN_PHASE2
except NameError:
    models_to_visualize = [ModelArch.MOBILENET_V3] # Fallback default

print("\n" + "★"*75)
print(f"🎨 MEMULAI RENDER VISUALISASI TRAINING UNTUK {len(models_to_visualize)} ARSITEKTUR")
print("★"*75)

config_vis = TrainingConfig()

for arch in models_to_visualize:
    try:
        # Inisialisasi visualizer untuk arsitektur spesifik
        visualizer = TrainingVisualizer(config=config_vis, architecture=arch)
        
        # Render dan simpan semua metrik
        visualizer.run_all_visualisations()
        
    except FileNotFoundError as e:
        print(e)
    except ValueError as e:
        print(e)
    except Exception as e:
        print(f"\n[CRITICAL ERROR] Gagal merender grafik untuk {arch.value}: {e}")

print("\n" + "★"*75)
print("🏆 SELURUH PROSES RENDER GRAFIK TRAINING SELESAI!")
print("★"*75)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🎨 MEMULAI RENDER VISUALISASI TRAINING UNTUK 1 ARSITEKTUR
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

📊 INISIALISASI VISUALIZER TRAINING: MobileNetV3Small
[ERROR] Database /kaggle/working/training_outputs/MobileNetV3Small/training_log_MobileNetV3Small.db tidak ditemukan.

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🏆 SELURUH PROSES RENDER GRAFIK TRAINING SELESAI!
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★


In [8]:
# ==============================================================================
# [CELL 8] DEBUG: INSPEKSI STRUKTUR DIREKTORI OUTPUT TRAINING
# ==============================================================================
import os

def print_directory_tree(startpath: str) -> None:
    """Mencetak struktur folder dan file artefak training."""
    print(f"\n🌳 STRUKTUR DIREKTORI TRAINING: {startpath}")
    print("="*60)
    
    if not os.path.exists(startpath):
        print(f"[WARNING] Direktori {startpath} belum dibuat.")
        return

    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * level
        print(f"{indent}📂 {os.path.basename(root)}/")
        
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            if f.endswith('.keras'):
                icon = "📦"
            elif f.endswith('.db'):
                icon = "🗄️"
            elif f.endswith('.png'):
                icon = "🖼️"
            else:
                icon = "📄"
            print(f"{subindent}{icon} {f}")

config_debug = TrainingConfig()
print_directory_tree(config_debug.BASE_OUTPUT_DIR)


🌳 STRUKTUR DIREKTORI TRAINING: /kaggle/working/training_outputs
📂 training_outputs/
    📂 MobileNetV3Small/
        📂 visualisation/
